# Aegis: Trust-Gated Support Agent — Demo

Aegis is a trust-gated customer support agent for B2B SaaS teams. Instead of sending AI-generated replies directly, it runs a multi-agent workflow that triages the ticket, looks up policy, drafts a response, judges it, assigns a trust score, and escalates risky cases to humans.

**Agentic concepts demonstrated in this notebook:**

1. **Multi-agent workflow** (triage -> policy -> draft -> judge -> reflection -> trust gate)
2. **Real MCP server** exposing a `policy_lookup` tool (`src/mcp_server.py`)
3. **Gemini-powered** drafting and judging with structured output (`--mode gemini`)
4. **Agent skills** — modular `SKILL.md` capabilities loaded per ticket category (`skills/`)
5. **Security/privacy trust gate** — high-risk cases are always escalated deterministically

## 1. Project Setup

Run this notebook from the project root. If you open it from `notebooks/`, the next cell moves up one level automatically. Make sure dependencies are installed first: `pip install -r requirements.txt`.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Current directory:", Path.cwd())
print("Key items:", [p.name for p in Path.cwd().iterdir() if p.name in ["src", "data", "policies", "skills", "README.md"]])

## 2. Load Sample Tickets

A small synthetic support-ticket dataset used for functional validation.

In [ ]:
import pandas as pd

df = pd.read_csv("data/sample_tickets.csv")
df

## 3. Run Aegis on One Ticket (rule mode)

The pipeline: Triage -> Policy Lookup -> Draft -> Judge -> Reflection -> Trust Gate.

In [ ]:
from src.main import process_ticket

ticket = "I was charged twice this month. Please refund me now."
result = process_ticket("demo-001", ticket, mode="rule")
result.model_dump()

## 4. Inspect the Draft Response

The draft is never trusted automatically — the judge evaluates it first.

In [ ]:
print(result.draft_response)

## 5. Trust Gate Decision

The gate routes to `suggest_reply`, `human_review`, or `escalate`.

In [ ]:
print("Category:", result.category)
print("Risk level:", result.risk_level)
print("Trust score:", result.trust_score)
print("Decision:", result.decision)
print("Reason:", result.reason)

if result.escalation_note:
    print()
    print("Escalation note:")
    print(result.escalation_note)

## 6. Security Ticket — Deterministic Escalation

Security and privacy tickets bypass reflection and are **always** escalated, regardless of model confidence. This is the core trust guarantee.

In [ ]:
security_ticket = "I found a security vulnerability that exposes API keys."
sec = process_ticket("demo-sec", security_ticket, mode="rule")
print("Category:", sec.category)
print("Decision:", sec.decision, "(expected: escalate)")
print("Trust score:", sec.trust_score)

## 7. Real MCP Server — Policy Lookup Tool

`src/mcp_server.py` exposes `policy_lookup` as a real MCP tool over stdio. Below we call the underlying tool directly. To see a live MCP client/server exchange, run `python -m src.mcp_client_demo` in a terminal.

In [ ]:
from src.tools import policy_lookup

print(policy_lookup("billing"))

## 8. Agent Skills

Modular `SKILL.md` capabilities are loaded per ticket category from `skills/`. Billing tickets load the refund-verification skill; security/privacy load the incident-response skill.

In [ ]:
from src.skills import list_skills, load_skill

print("Available skills:", list_skills())
print()
print(load_skill("billing")[:400], "...")

## 9. Gemini Mode (optional)

If a `GEMINI_API_KEY` is set (in `.env` or Kaggle Secrets), the same pipeline runs with Gemini-powered drafting and judging. Otherwise this cell is skipped so the notebook stays reproducible.

In [ ]:
from src.config import settings

if settings.gemini_api_key:
    g = process_ticket("demo-gemini", ticket, mode="gemini")
    print("Gemini decision:", g.decision, "| trust:", g.trust_score)
    print()
    print(g.draft_response)
else:
    print("No GEMINI_API_KEY found — skipping Gemini mode.")
    print("Set GEMINI_API_KEY and GEMINI_MODEL in .env to run this cell.")

## 10. Run All Sample Tickets

In [ ]:
results = []
for _, row in df.iterrows():
    r = process_ticket(row["ticket_id"], row["customer_message"], mode="rule")
    results.append(r.model_dump())

results_df = pd.DataFrame(results)
results_df[["ticket_id", "category", "risk_level", "trust_score", "decision", "reason"]]

## 11. Evaluation Harness

Compares Aegis predictions against expected labels in `data/sample_tickets.csv`.

In [ ]:
from src.evaluator import evaluate_sample

report = evaluate_sample("data/sample_tickets.csv")
report["metrics"]

## 12. Per-Ticket Evaluation Results

In [ ]:
eval_df = pd.DataFrame(report["rows"])
eval_df[[
    "ticket_id",
    "expected_category",
    "predicted_category",
    "category_match",
    "expected_risk",
    "predicted_risk",
    "risk_match",
    "expected_decision",
    "predicted_decision",
    "decision_match",
    "trust_score",
]]

## 13. Concept Summary

- **Multi-agent workflow:** triage, draft, judge, reflection, and escalation agents pass typed Pydantic messages (`TriageResult -> DraftResult -> JudgeResult -> FinalResult`).
- **Real MCP server:** `src/mcp_server.py` exposes `policy_lookup` over stdio (client demo: `src/mcp_client_demo.py`).
- **Gemini-powered:** drafting and judging use Gemini with structured output in `--mode gemini`.
- **Agent skills:** modular `SKILL.md` files loaded per category from `skills/`.
- **Trust gate:** security/privacy cases are always escalated; low-trust drafts are revised and re-judged before routing.

## 14. Notes & Limitations

- The sample dataset is small and synthetic, so metrics are a functional validation of the workflow, not a production benchmark.
- Policy lookup reads local Markdown files; a production deployment would connect the MCP server to a real knowledge base, CRM, or ticketing system.
- Rule mode is fully deterministic and needs no API key; Gemini mode requires a valid key and model.